This notebook loads the original data files into better organized data structure with onyl relevant information + allows querying and normalization tools on that dataset

In [ ]:
pip install pyarrow dask[parquet]

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pip.lynx.md/repository/lynx/simple
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import re
import ast
import json
import gc
import shutil
import pandas as pd
from pandas import Timedelta
import dask.dataframe as dd
import dask
from dask.diagnostics import ProgressBar
from datetime import datetime, timedelta, time
import pytz
import numpy as np
from tqdm import tqdm
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Union, Any
from collections import defaultdict

from dask.distributed import Client

dask.config.set({
    "distributed.worker.memory.spill": False,
    "dataframe.shuffle.method": "tasks",   # prevent drop_duplicates/merge from using P2P
})

client = Client(processes=True)

base_path = Path("/home/jovyan/workspace/data")

RAW_ROOT_OLD = base_path / "original-data" / "pre-intervention"
RAW_ROOT_NEW = base_path / "original-data" / "post-intervention"
OUT_ROOT = base_path / "processed-data"
DIM_ROOT = OUT_ROOT / "dimensions"
RULES_DIR = OUT_ROOT / "normalization_queries"

OUT_ROOT.mkdir(parents=True, exist_ok=True)
DIM_ROOT.mkdir(parents=True, exist_ok=True)
RULES_DIR.mkdir(parents=True, exist_ok=True)


# Build Processed Tables

## Mappings

In [ ]:
# -------------------------------------------------------------------
# 1. Define mappings from each raw OMOP table to unified schemas
#    Fill these dicts with your real column names.
# -------------------------------------------------------------------

# Condition -> clinical_events
CONDITION_MAPPING = {
    # "origin": "condition"
    "person_id": "person_id",
    "visit_occurrence_id": "visit_id",
    "condition_start_datetime": "start_datetime",
    "condition_end_datetime": "end_datetime", # if missing, start datetime + 1s
    "condition_concept_id": "concept_id",
    "condition_concept_name": "concept_name",
    "condition_source_value": "source_value",
}

OBSERVATION_MAPPING = {
    # "origin": "observation"
    "person_id": "person_id",
    "visit_occurrence_id": "visit_id",
    "observation_datetime": "start_datetime",
    # Fill end_datetime with start time + 1s
    "observation_concept_id": "concept_id",
    "observation_concept_name": "concept_name",
    "observation_source_value": "source_value",
    "value_as_string": "str_value",
    "value_as_number": "value",
}

PROCEDURE_MAPPING = {
    # "origin": "procedure"
    "person_id": "person_id",
    "visit_occurrence_id": "visit_id",
    "procedure_datetime": "start_datetime",
    # Fill end_datetime with start time + 1s
    "procedure_concept_id": "concept_id",
    "procedure_concept_name": "concept_name",
    "procedure_source_value": "source_value",
}

DEVICE_MAPPING = {
    # "origin": "device_exposure"
    "person_id": "person_id",
    "visit_occurrence_id": "visit_id",
    "device_exposure_start_datetime": "start_datetime",
    "device_exposure_end_datetime": "end_datetime", # if missing, start datetime + 1s
    "device_concept_id": "concept_id",
    "device_concept_name": "concept_name",
    "device_source_value": "source_value",
}

MEASUREMENT_EVENTS_MAPPING = { # Nominal measurements Added to clinical events
    "person_id": "person_id",
    "visit_occurrence_id": "visit_id",
    "measurement_datetime": "start_datetime",
    # Fill end_datetime with start time + 1s
    "measurement_concept_id": "concept_id",
    "measurement_concept_name": "concept_name",
    "measurement_source_value": "source_value",
}

VISIT_MAPPING = {
    "person_id": "person_id",
    "visit_occurrence_id": "visit_id",
    "visit_start_datetime": "start_datetime",
    "visit_end_datetime": "end_datetime",
    "visit_concept_name": "visit_type",
}

DEATH_MAPPING = { # Merged with VISIT_MAPPING
    "person_id": "person_id",
    "death_datetime": "death_datetime", # Can be Null
    # Indication if death is within 30 days from visit end_time
}

PERSON_MAPPING = { # Merged with VISIT_MAPPING
    "person_id": "person_id",
    "gender_concept_name": "gender",
    "birth_datetime": "birth_datetime",
}

VISIT_TYPE_MAPPING = { # Seperate Table
    "person_id": "person_id",
    "visit_occurrence_id": "visit_id",
    "visit_detail_concept_name": "visit_type",
    "care_site_name": "care_site_name",
}

DRUG_MAPPING = {
    "person_id": "person_id",
    "visit_occurrence_id": "visit_id",
    "drug_exposure_start_datetime": "start_datetime",
    "drug_exposure_end_datetime": "end_datetime",
    "drug_type_concept_id": "drug_type_concept_id",
    # "drug_type_concept_name": "drug_type_concept_name", -> manual mapping
    "drug_concept_id": "concept_id",
    "drug_source_value": "source_code",
    "drug_concept_name": "concept_name",
    "route_concept_name": "route",
    "route_source_value": "route_source",
    "quantity": "quantity",
    "days_supply": "days_supply",
    "dose_unit_source_value": "dose_unit",
}

DRUG_TO_ATC_MAPPING = { # Merged with DRUG_MAPPING
    "SOURCE_CODE": "source_code",
    "SOURCE_NAME": "source_name",
    "ATC": "atc",
}

MEASUREMENT_MAPPING = { # Seperate Table
    "object_type": "object_type",
    "visit_occurrence_id": "visit_id",
    "person_id": "person_id",
    "measurement_datetime": "start_datetime",
    # Fill end_datetime with start time + 1s
    "measurement_concept_id": "concept_id",
    "measurement_concept_name": "concept_name",
    "value_as_number": "value",
    "unit_source_value": "unit",
}

# Dedup subsets
CLINICAL_DEDUP_COLS = [
    "person_id", "visit_id", "start_datetime", "end_datetime",
    "concept_id", "origin", "hospital"
]

VISIT_DEDUP_COLS = [
    "person_id", "visit_id", "start_datetime", "end_datetime", "hospital"
]

DRUG_DEDUP_COLS = [
    "person_id", "visit_id", "start_datetime", "concept_id",
    "drug_type_category", "hospital", "route"
]

MEAS_DEDUP_COLS = [
    "person_id", "visit_id", "start_datetime", "concept_id", "hospital"
]

VISIT_TYPE_DEDUP_COLS = [
    "person_id", "visit_id", "visit_type", "care_site_name", "hospital"
]


# Drug type concept ? category
DRUG_TYPE_CATEGORY = {
    "32818": "given_in_hospital",
    "32833": "given_in_hospital",
    "32838": "discharge_prescription",
    "32865": "chronic_medications",
}

In [ ]:
# Hospital name IS the folder name in the new structure (e.g. pre-intervention/Barzilai/)
def extract_hospital_name(hospital_dir_name):
    return hospital_dir_name

def find_table_folders(hospital_dir, table_prefix):
    """
    Return folders inside a hospital directory whose names start with table_prefix.
    Works for both flat naming (condition/) and suffixed naming (condition-BARZILAI-abc/).
    """
    matches = []
    for sub in hospital_dir.iterdir():
        if sub.is_dir() and sub.name.lower().startswith(table_prefix.lower()):
            matches.append(sub)
    return matches

def inspect_raw_columns(table_prefix, max_hospitals=1, raw_roots=None):
    if raw_roots is None:
        raw_roots = [RAW_ROOT_OLD, RAW_ROOT_NEW]

    seen = 0
    for raw_root in raw_roots:
        hospital_dirs = [p for p in raw_root.iterdir() if p.is_dir()]
        for hdir in hospital_dirs:
            if seen >= max_hospitals:
                return
            hospital = extract_hospital_name(hdir.name)
            table_dirs = find_table_folders(hdir, table_prefix)
            if not table_dirs:
                print(f"[{hospital}] No folder starting with '{table_prefix}'")
                continue
            for tdir in table_dirs:
                parquet_files = [str(p) for p in tdir.rglob("*.parquet")]
                if not parquet_files:
                    print(f"[{hospital}] {tdir.name} has no parquet files")
                    continue
                ddf = dd.read_parquet(parquet_files, engine="pyarrow")
                print(f"\n=== {hospital} :: {tdir.name} ===")
                print(list(ddf.columns))
            seen += 1

# inspect_raw_columns("measurement")


In [ ]:
def read_raw_table(table_prefix: str, raw_roots=None, hospitals=None, **read_kwargs) -> dd.DataFrame:
    """
    Read all parquet files for a given table prefix from both pre- and post-intervention
    source roots, concatenating them into a single Dask DataFrame.
    Extra kwargs are forwarded to dd.read_parquet (e.g. split_row_groups, blocksize).
    hospitals: optional set/list to restrict which hospitals are read.
    """
    if raw_roots is None:
        raw_roots = [RAW_ROOT_OLD, RAW_ROOT_NEW]

    dfs = []
    for raw_root in raw_roots:
        hospital_dirs = [p for p in raw_root.iterdir() if p.is_dir()]
        for hdir in hospital_dirs:
            hospital = extract_hospital_name(hdir.name)
            if hospitals is not None and hospital not in hospitals:
                continue
            table_dirs = find_table_folders(hdir, table_prefix)
            if not table_dirs:
                continue
            for tdir in table_dirs:
                parquet_files = [str(p) for p in tdir.rglob("*.parquet")]
                if not parquet_files:
                    continue
                ddf = dd.read_parquet(parquet_files, engine="pyarrow", **read_kwargs)
                # Normalize to lowercase â€” NEW source exports UPPERCASE column names
                col_map = {c: c.lower() for c in ddf.columns if c != c.lower()}
                if col_map:
                    ddf = ddf.rename(columns=col_map)
                ddf = ddf.assign(hospital=hospital)
                dfs.append(ddf)

    if not dfs:
        raise ValueError(f"No parquet files found for prefix '{table_prefix}'")

    return dd.concat(dfs, axis=0, interleave_partitions=True)


def map_to_unified(ddf: dd.DataFrame, mapping: dict, constant_cols: dict = None) -> dd.DataFrame:
    """
    Select and rename columns according to mapping.
    Always keeps 'hospital'. Adds any constant columns.
    """
    base_cols = list(mapping.keys())
    ddf = ddf[base_cols + ["hospital"]]
    ddf = ddf.rename(columns=mapping)

    if constant_cols:
        for k, v in constant_cols.items():
            ddf[k] = v

    return ddf


def _normalize_datetime_series(s: pd.Series) -> pd.Series:
    """Convert any datetime-like series to timezone-naive datetime64[ns]."""
    s = pd.to_datetime(s, errors="coerce", utc=True)
    s = s.dt.tz_convert(None)
    return s


def fix_end_datetime(ddf: dd.DataFrame) -> dd.DataFrame:
    """
    If end_datetime is missing or <= start_datetime, set it to start_datetime + 1s.
    """
    def _fix(pdf: pd.DataFrame) -> pd.DataFrame:
        pdf = pdf.copy()
        if "start_datetime" not in pdf.columns:
            raise KeyError("start_datetime column missing")
        if "end_datetime" not in pdf.columns:
            pdf["end_datetime"] = pd.NaT
        pdf["start_datetime"] = _normalize_datetime_series(pdf["start_datetime"])
        pdf["end_datetime"]   = _normalize_datetime_series(pdf["end_datetime"])
        mask = pdf["end_datetime"].isna() | (pdf["end_datetime"] <= pdf["start_datetime"])
        pdf.loc[mask, "end_datetime"] = pdf.loc[mask, "start_datetime"] + Timedelta(seconds=1)
        return pdf

    return ddf.map_partitions(_fix)


def filter_invalid_concepts(ddf: dd.DataFrame, concept_col: str = "concept_name") -> dd.DataFrame:
    """Remove rows where concept_name contains 'no matching_concept' variants."""
    if concept_col not in ddf.columns:
        return ddf
    bad_patterns = ["no matching_concept", "no_matching_concept", "no matching concept"]
    expr = "(?i)" + "|".join(bad_patterns)
    return ddf[~ddf[concept_col].astype(str).str.contains(expr, regex=True)]


def enforce_unique_concept_ids(ddf: dd.DataFrame) -> dd.DataFrame:
    """
    Ensure each concept_id maps to exactly one concept_name.
    If a concept_id appears with multiple names, suffix duplicates.

    Strategy: each partition locally deduplicates its (concept_id, concept_name) pairs
    (tiny output), then we collect all local pairs to the driver and do the conflict
    detection in pandas. Zero shuffles.

    Uses synchronous scheduler for the internal .compute() â€” avoids sending large
    measurement partitions to distributed workers which OOM and deadlock indefinitely.
    """
    pairs = (
        ddf[["concept_id", "concept_name"]]
        .map_partitions(lambda pdf: pdf.drop_duplicates())
        .compute(scheduler="synchronous")  # one partition at a time â€” safe on large tables
        .drop_duplicates()
    )

    counts = pairs.groupby("concept_id")["concept_name"].nunique()
    need_fix_ids = counts[counts > 1].index.tolist()

    if not need_fix_ids:
        return ddf

    suffix_map = {}
    for cid in need_fix_ids:
        names = pairs.loc[pairs["concept_id"] == cid, "concept_name"].unique()
        for i, name in enumerate(names, start=1):
            new_id = str(cid) if i == 1 else f"{cid}__{i}"
            suffix_map[(str(cid), name)] = new_id

    need_fix_set = set(need_fix_ids)

    def _apply(pdf: pd.DataFrame) -> pd.DataFrame:
        pdf = pdf.copy()
        mask = pdf["concept_id"].isin(need_fix_set)
        if not mask.any():
            return pdf
        pdf.loc[mask, "concept_id"] = [
            suffix_map.get((str(cid), name), str(cid))
            for cid, name in zip(pdf.loc[mask, "concept_id"], pdf.loc[mask, "concept_name"])
        ]
        return pdf

    return ddf.map_partitions(_apply)


def coerce_datetimes(ddf: dd.DataFrame, dt_cols) -> dd.DataFrame:
    """Coerce listed columns to tz-naive datetime64[ns]."""
    for col in dt_cols:
        if col in ddf.columns:
            ddf[col] = dd.to_datetime(ddf[col], errors="coerce", utc=True).dt.tz_convert(None)
    return ddf


def coerce_numerics(ddf: dd.DataFrame, numeric_cols, dtype: str = "float64") -> dd.DataFrame:
    """Coerce listed columns to numeric with consistent dtype across partitions."""
    for col in numeric_cols:
        if col in ddf.columns:
            ddf[col] = dd.to_numeric(ddf[col], errors="coerce").astype(dtype)
    return ddf


def coerce_strings(ddf: dd.DataFrame, string_cols) -> dd.DataFrame:
    """Force listed columns to pandas nullable string dtype across all partitions."""
    for col in string_cols:
        if col in ddf.columns:
            ddf[col] = ddf[col].astype("string")
    return ddf


def repartition(ddf: dd.DataFrame, npartitions: int = 16) -> dd.DataFrame:
    """Simple repartition without scanning memory usage."""
    return ddf.repartition(npartitions=npartitions)


def _write_partitions(ddf, out_dir, label="", partition_on=None, sort_cols=None, row_group_size=None):
    """
    Write a Dask DataFrame partition-by-partition with explicit gc.collect().

    partition_on : list[str], optional
        Hive-style partitioning (single column), e.g. ["hospital"]. Lets
        downstream readers skip whole subfolders on an equality/IN filter for
        that column, on top of row-group pruning -- both filters compose.
    sort_cols : list[str], optional
        Sort each written chunk before writing so row-group min/max stats are
        tight enough for pruning on these columns too (e.g. concept_id). This
        is a local sort per chunk (per original Dask partition x hospital),
        not a global sort across the whole table -- a global sort would need
        a shuffle, which we avoid given spill is disabled.
    row_group_size : int, optional
        Smaller row groups make stats-based pruning meaningful; a single huge
        row group per file (the old default) makes pruning a no-op.

    Resume support is tracked via marker files under out_dir/.write_done/,
    since output paths now vary by partition value when partition_on is set.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    done_marker_dir = out_dir / ".write_done"
    done_marker_dir.mkdir(exist_ok=True)

    delayed_parts = ddf.to_delayed()
    total = len(delayed_parts)  # authoritative: npartitions can differ in dask-expr due to fusion
    already = {int(p.stem) for p in done_marker_dir.glob("*.done") if int(p.stem) < total}
    remaining = [i for i in range(total) if i not in already]

    pq_kwargs = {"engine": "pyarrow", "index": False}
    if row_group_size:
        pq_kwargs["row_group_size"] = row_group_size

    desc = f"[{label}]" if label else "partitions"
    with tqdm(remaining, desc=desc, initial=len(already), total=total, unit="part") as pbar:
        for i in remaining:
            pdf = None
            try:
                pdf = delayed_parts[i].compute(scheduler="synchronous")

                if sort_cols:
                    pdf = pdf.sort_values([c for c in sort_cols if c in pdf.columns])

                if partition_on:
                    col = partition_on[0]
                    for val, sub in pdf.groupby(col, observed=True):
                        sub_dir = out_dir / f"{col}={val}"
                        sub_dir.mkdir(parents=True, exist_ok=True)
                        sub.drop(columns=[col]).to_parquet(sub_dir / f"part.{i:04d}.parquet", **pq_kwargs)
                else:
                    pdf.to_parquet(out_dir / f"part.{i:04d}.parquet", **pq_kwargs)

                (done_marker_dir / f"{i}.done").touch()
            except Exception as e:
                tqdm.write(f"  [{label}] ERROR on partition {i}: {e}")
                raise
            finally:
                del pdf
                gc.collect()

            pbar.update(1)


def summarize_table(name: str, ddf, key_cols=None, max_unique: int = 10):
    """Quick summary: row count, dtypes, null counts, sample values."""
    print(f"\n===== SUMMARY: {name} =====")
    n_rows = int(ddf.shape[0].compute())
    print(f"Rows: {n_rows}")
    print("\nDtypes:")
    print(ddf.dtypes)
    for col in (key_cols or []):
        if col not in ddf.columns:
            print(f"\n[WARN] Column {col} not in table")
            continue
        nulls = ddf[col].isna().sum().compute()
        print(f"\nColumn: {col}")
        print(f"  nulls: {nulls}")
        try:
            s = ddf[col].dropna().drop_duplicates().head(max_unique)
            if hasattr(s, "compute"):
                s = s.compute()
            print(f"  sample values: {s.tolist()}")
        except Exception as e:
            print(f"  could not get sample values: {e}")


In [ ]:
EXPECTED_HOSPITALS = {"Barzilai", "Bz", "Gmc", "Hy", "Poria", "Shamir"}
# New source data extends into 2025; any max date before this is a sign the
# post-intervention source was not ingested.
MIN_EXPECTED_MAX_DATE = pd.Timestamp("2024-08-01")


def validate_visits(out_root=OUT_ROOT):
    """
    Sanity-check the written visits table:
      - All expected hospitals present
      - Max start_datetime per hospital is beyond the old-data cutoff (July 2024)
      - Row counts are non-trivial
    Raises AssertionError on any failure so the pipeline stops early.
    """
    print("\n=== Validating visits ===")
    df = pd.read_parquet(str(out_root / "visits"),
                         columns=["hospital", "start_datetime", "visit_id"])
    df["start_datetime"] = pd.to_datetime(df["start_datetime"], errors="coerce")

    hospitals_found = set(df["hospital"].unique())
    missing = EXPECTED_HOSPITALS - hospitals_found
    assert not missing, f"Missing hospitals in visits: {missing}"
    print(f"  Hospitals OK: {sorted(hospitals_found)}")

    problems = []
    for h, g in df.groupby("hospital", observed=True):
        max_date = g["start_datetime"].max()
        n = len(g)
        status = "OK" if max_date >= MIN_EXPECTED_MAX_DATE else "STALE"
        print(f"  {h}: {n:,} rows, max date {max_date.date()}  [{status}]")
        if max_date < MIN_EXPECTED_MAX_DATE:
            problems.append(f"{h} max_date={max_date.date()} < {MIN_EXPECTED_MAX_DATE.date()}")

    assert not problems, "New data missing from visits:\n  " + "\n  ".join(problems)
    print("  visits validation PASSED")


def validate_visit_types(out_root=OUT_ROOT):
    """
    Sanity-check the written visit_types table:
      - All expected hospitals present
      - care_site_name is not entirely null
      - visit_ids are a subset of those in visits (inner-join integrity)
    Raises AssertionError on any failure.
    """
    print("\n=== Validating visit_types ===")
    vt = pd.read_parquet(str(out_root / "visit_types"),
                         columns=["hospital", "visit_id", "care_site_name"])
    visits_ids = pd.read_parquet(str(out_root / "visits"),
                                 columns=["visit_id"])["visit_id"]

    hospitals_found = set(vt["hospital"].unique())
    missing = EXPECTED_HOSPITALS - hospitals_found
    assert not missing, f"Missing hospitals in visit_types: {missing}"
    print(f"  Hospitals OK: {sorted(hospitals_found)}")

    null_pct = vt["care_site_name"].isna().mean() * 100
    assert null_pct < 50, f"care_site_name is {null_pct:.1f}% null â€” likely a mapping issue"
    print(f"  care_site_name null rate: {null_pct:.1f}%  OK")

    orphan_mask = ~vt["visit_id"].isin(visits_ids)
    orphan_pct = orphan_mask.mean() * 100
    assert orphan_pct < 1, f"{orphan_pct:.1f}% of visit_type rows have no matching visit_id"
    print(f"  Orphan visit_ids: {orphan_pct:.2f}%  OK")

    for h, g in vt.groupby("hospital", observed=True):
        print(f"  {h}: {len(g):,} rows")

    print("  visit_types validation PASSED")


## Build Clinical Events Table

In [ ]:
def build_clinical_events():
    _TMP = OUT_ROOT / "_tmp_clinical"
    _TMP.mkdir(parents=True, exist_ok=True)

    # â”€â”€ Phase 1: write each sub-table to its own checkpoint folder â”€â”€â”€â”€â”€â”€â”€
    # One compute at a time. meas_evt uses synchronous scheduler (largest source table).

    for name, prefix, mapping, constants in [
        ("cond", "condition",            CONDITION_MAPPING,          {"origin": "condition"}),
        ("obs",  "observation",          OBSERVATION_MAPPING,        {"origin": "observation"}),
        ("proc", "procedure_occurrence", PROCEDURE_MAPPING,          {"origin": "procedure"}),
        ("dev",  "device_exposure",      DEVICE_MAPPING,             {"origin": "device_exposure"}),
    ]:
        raw = read_raw_table(prefix)
        df  = map_to_unified(raw, mapping, constants)
        df  = coerce_datetimes(df, ["start_datetime", "end_datetime"])
        if name == "obs":
            df = coerce_numerics(df, ["value"])
        if "value" not in df.columns:
            df["value"] = np.nan
        if "str_value" not in df.columns:
            df["str_value"] = "NULL"
        df = fix_end_datetime(df)
        df = filter_invalid_concepts(df)
        df.to_parquet(_TMP / name, engine="pyarrow", write_index=False)
        del raw, df
        gc.collect()
        print(f"  [checkpoint] {name} written")

    # measurement events: column-pruned + culture filter + synchronous write
    meas_raw = read_raw_table("measurement")
    _needed = [c for c in list(MEASUREMENT_EVENTS_MAPPING.keys()) if c in meas_raw.columns]
    meas_raw = meas_raw[_needed + ["hospital"]]
    nc = meas_raw["measurement_concept_name"].str.lower()
    meas_raw = meas_raw[
        (nc.str.contains("blood") & nc.str.contains("culture")) |
        (nc.str.contains("urine") & nc.str.contains("culture"))
    ]
    meas_evt = map_to_unified(meas_raw, MEASUREMENT_EVENTS_MAPPING, {"origin": "measurement_nominal"})
    meas_evt = coerce_datetimes(meas_evt, ["start_datetime", "end_datetime"])
    if "value" not in meas_evt.columns:
        meas_evt["value"] = np.nan
    if "str_value" not in meas_evt.columns:
        meas_evt["str_value"] = "NULL"
    meas_evt = fix_end_datetime(meas_evt)
    meas_evt = filter_invalid_concepts(meas_evt)
    meas_evt.to_parquet(
        _TMP / "meas_evt", engine="pyarrow", write_index=False,
        compute_kwargs={"scheduler": "synchronous"},
    )
    del meas_raw, meas_evt
    gc.collect()
    print("  [checkpoint] meas_evt written")

    # â”€â”€ Phase 2: concat from checkpoints, dedup, filter, write â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # All operations are partition-local (no shuffle).
    # Final write uses synchronous scheduler â€” avoids large-graph deadlock
    # caused by embedding visits_keys_pd in distributed task graph.

    clinical = dd.concat([
        dd.read_parquet(_TMP / n, engine="pyarrow")
        for n in ["cond", "obs", "proc", "dev", "meas_evt"]
    ], axis=0, interleave_partitions=True)

    dedup_cols = [c for c in CLINICAL_DEDUP_COLS if c in clinical.columns]
    clinical = clinical.map_partitions(lambda pdf: pdf.drop_duplicates(subset=dedup_cols))

    clinical = enforce_unique_concept_ids(clinical)

    clinical = coerce_numerics(clinical, ["value"])
    clinical = coerce_strings(clinical, ["str_value"])
    clinical = clinical.dropna(subset=["person_id", "visit_id", "concept_name",
                                        "start_datetime", "end_datetime"])

    # Load visits keys once â€” synchronous scheduler processes partitions sequentially
    # so the pandas DataFrame stays in driver memory, not serialized into task graph
    visits_keys_pd = pd.read_parquet(
        str(OUT_ROOT / "visits"),
        columns=["person_id", "visit_id", "hospital"],
    ).drop_duplicates()

    def _filter_by_visits(pdf):
        return pdf.merge(visits_keys_pd, on=["person_id", "visit_id", "hospital"], how="inner")

    clinical = clinical.map_partitions(_filter_by_visits)

    out_path = OUT_ROOT / "clinical_events"
    # Hive-partition by hospital + sort by concept_id + small row groups, so
    # get_data()-style queries can prune both whole hospital folders and
    # row groups within a file.
    _write_partitions(
        clinical, out_path, label="clinical_events",
        partition_on=["hospital"], sort_cols=["concept_id"], row_group_size=50_000,
    )
    shutil.rmtree(_TMP, ignore_errors=True)
    print(f"clinical_events written to {out_path}")

## Build Visits Table

In [ ]:
def build_visits():
    visits_raw = read_raw_table("visit_occurrence")
    death_raw  = read_raw_table("death")
    person_raw = read_raw_table("person")

    visits = map_to_unified(visits_raw, VISIT_MAPPING)
    death  = map_to_unified(death_raw,  DEATH_MAPPING)
    person = map_to_unified(person_raw, PERSON_MAPPING)

    visits = coerce_datetimes(visits, ["start_datetime", "end_datetime"])
    death  = coerce_datetimes(death,  ["death_datetime"])
    person = coerce_datetimes(person, ["birth_datetime"])

    visits = fix_end_datetime(visits)

    # Synchronous scheduler: avoids sending large task graph to distributed workers.
    death_pd = (
        death[["person_id", "death_datetime", "hospital"]]
        .compute(scheduler="synchronous")
        .groupby(["person_id", "hospital"], as_index=False)
        .agg({"death_datetime": lambda x: x.mode().iloc[0] if x.notna().any() else pd.NaT})
    )
    person_pd = (
        person[["person_id", "gender", "birth_datetime", "hospital"]]
        .compute(scheduler="synchronous")
        .groupby(["person_id", "hospital"], as_index=False)
        .agg({
            "gender":         lambda x: x.mode().iloc[0],
            "birth_datetime": lambda x: x.mode().iloc[0],
        })
    )

    def _merge_broadcast(pdf):
        pdf = pdf.merge(death_pd,  on=["person_id", "hospital"], how="left")
        pdf = pdf.merge(person_pd, on=["person_id", "hospital"], how="left")
        pdf["death_flag"] = pdf["death_datetime"].notnull()
        return pdf

    visits = visits.map_partitions(_merge_broadcast)

    # Per-partition dedup avoids a shuffle that makes npartitions inconsistent
    # between ddf.npartitions and to_delayed() in the dask-expr backend.
    # Cross-partition duplicates are rare for OMOP visit_occurrence data.
    dedup_cols = [c for c in VISIT_DEDUP_COLS if c in visits.columns]
    visits = visits.map_partitions(lambda pdf: pdf.drop_duplicates(subset=dedup_cols))

    visits = visits.dropna(subset=[
        "person_id", "visit_id", "start_datetime", "end_datetime",
        "visit_type", "gender", "death_flag", "birth_datetime"
    ])

    out_path = OUT_ROOT / "visits"
    _write_partitions(visits, out_path, label="visits",
                      partition_on=["hospital"], sort_cols=["visit_id"], row_group_size=50_000)


## Build Visit Type Table

In [ ]:
def build_visit_types():
    """
    Build a small visit_types table from visit_detail:
    person_id, visit_id, visit_type, care_site_name, hospital
    """
    detail_raw = read_raw_table("visit_detail")
    vtypes = map_to_unified(detail_raw, VISIT_TYPE_MAPPING)
    vtypes = filter_invalid_concepts(vtypes, concept_col="visit_type")

    # Within-partition dedup â€” no P2P shuffle
    dedup_cols = [c for c in VISIT_TYPE_DEDUP_COLS if c in vtypes.columns]
    vtypes = vtypes.map_partitions(lambda pdf: pdf.drop_duplicates(subset=dedup_cols))

    vtypes = vtypes.dropna(subset=["care_site_name", "visit_id"])

    # Broadcast join â€” no shuffle.
    # pd.read_parquet on a Hive-partitioned directory reconstructs the hospital
    # partition column from folder names automatically via the pyarrow engine.
    visits_keys_pd = pd.read_parquet(
        str(OUT_ROOT / "visits"),
        columns=["person_id", "visit_id", "hospital"],
    ).drop_duplicates()

    def _filter_by_visits(pdf):
        return pdf.merge(visits_keys_pd, on=["person_id", "visit_id", "hospital"], how="inner")

    vtypes = vtypes.map_partitions(_filter_by_visits)

    out_path = OUT_ROOT / "visit_types"
    _write_partitions(vtypes, out_path, label="visit_types",
                      partition_on=["hospital"], sort_cols=["visit_id"], row_group_size=50_000)


## Build Drugs Table

In [ ]:
def build_drug_exposure():
    _TMP = OUT_ROOT / "_tmp_drug"
    _TMP.mkdir(parents=True, exist_ok=True)

    # â”€â”€ Phase 1: read + coerce + ATC merge + write checkpoint â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

    drug_raw = read_raw_table("drug_exposure")
    drug = map_to_unified(drug_raw, DRUG_MAPPING)
    drug = coerce_datetimes(drug, ["start_datetime", "end_datetime"])
    drug = coerce_numerics(drug, ["quantity", "days_supply"])
    drug = fix_end_datetime(drug)
    drug = filter_invalid_concepts(drug, concept_col="concept_name")

    def _map_drug_type(pdf):
        pdf = pdf.copy()
        s = pdf["drug_type_concept_id"].astype(str)
        pdf["drug_type_category"] = s.map(DRUG_TYPE_CATEGORY).fillna("other")
        return pdf

    drug = drug.map_partitions(_map_drug_type)

    try:
        atc_raw = read_raw_table("drug_to_atc")
        # read_raw_table normalizes cols to lowercase; apply same to mapping keys
        _atc_map = {k.lower(): v for k, v in DRUG_TO_ATC_MAPPING.items()}
        atc = atc_raw[list(_atc_map.keys())].rename(columns=_atc_map)
        atc_pd = atc.compute(scheduler="synchronous").drop_duplicates(subset=["source_code"])
        def _merge_atc(pdf):
            return pdf.merge(atc_pd, on="source_code", how="left")
        drug = drug.map_partitions(_merge_atc)
    except ValueError:
        print("No drug_to_atc table available.")

    drug.to_parquet(_TMP, engine="pyarrow", write_index=False,
                    compute_kwargs={"scheduler": "synchronous"})
    del drug_raw, drug
    gc.collect()
    print("[drug_exposure] checkpoint written")

    # â”€â”€ Phase 2: dedup + filter + write â€” shuffle-free â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

    drug = dd.read_parquet(_TMP, engine="pyarrow")

    dedup_cols = [c for c in DRUG_DEDUP_COLS if c in drug.columns]
    drug = drug.map_partitions(lambda pdf: pdf.drop_duplicates(subset=dedup_cols))

    drug = enforce_unique_concept_ids(drug)

    drug = coerce_numerics(drug, ["quantity", "days_supply"])
    drug = drug.dropna(subset=["person_id", "visit_id", "concept_name",
                                "start_datetime", "end_datetime"])

    visits_keys_pd = pd.read_parquet(
        str(OUT_ROOT / "visits"),
        columns=["person_id", "visit_id", "hospital"],
    ).drop_duplicates()

    def _filter_by_visits(pdf):
        return pdf.merge(visits_keys_pd, on=["person_id", "visit_id", "hospital"], how="inner")

    drug = drug.map_partitions(_filter_by_visits)

    out_path = OUT_ROOT / "drug_exposure"
    _write_partitions(
        drug, out_path, label="drug_exposure",
        partition_on=["hospital"], sort_cols=["concept_id"], row_group_size=50_000,
    )
    shutil.rmtree(_TMP, ignore_errors=True)
    print(f"drug_exposure written to {out_path}")


## Build Measurements Table

In [ ]:
def build_measurement():
    global client
    _TMP = OUT_ROOT / "_tmp_measurement"

    # â”€â”€ Phase 1 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    # split_row_groups=True: Dask splits large source parquet files at row group
    # boundaries instead of one partition per file â€” prevents single large hospital
    # files from creating giant partitions that OOM the driver.
    print("[measurement] Phase 1 starting (hospital-by-hospital)...")
    _TMP.mkdir(parents=True, exist_ok=True)
    # Close distributed client for Phase 1 — otherwise it intercepts
    # scheduler='synchronous' and routes work to workers that OOM and deadlock.
    client.close()
    for _hosp in sorted(EXPECTED_HOSPITALS):
        _hosp_done = _TMP / f".done_{_hosp}"
        if _hosp_done.exists():
            print(f"  [Phase 1/{_hosp}] already done, skipping")
            continue
        print(f"  [Phase 1/{_hosp}] reading...")
        meas_raw = read_raw_table("measurement", split_row_groups=True, blocksize="32MB",
                                   hospitals={_hosp})
        meas = map_to_unified(meas_raw, MEASUREMENT_MAPPING)
        meas = coerce_datetimes(meas, ["start_datetime"])
        meas = coerce_numerics(meas, ["value"])
        meas = fix_end_datetime(meas)
        meas = filter_invalid_concepts(meas)
        _write_partitions(meas, _TMP / _hosp, label=f"Phase 1/{_hosp}")
        del meas_raw, meas
        gc.collect()
        _hosp_done.touch()
        print(f"  [Phase 1/{_hosp}] done")
    print("[measurement] Phase 1 complete")

    # â”€â”€ Phase 2 â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print("[measurement] Phase 2 starting...")
    meas = dd.read_parquet(_TMP, engine="pyarrow")

    dedup_cols = [c for c in MEAS_DEDUP_COLS if c in meas.columns]
    meas = meas.map_partitions(lambda pdf: pdf.drop_duplicates(subset=dedup_cols))

    print("[measurement] Scanning concept_id uniqueness...")
    meas = enforce_unique_concept_ids(meas)

    meas = coerce_numerics(meas, ["value"])
    meas = coerce_strings(meas, ["unit"])
    meas = meas.dropna(subset=["person_id", "visit_id", "concept_name", "value",
                                "start_datetime", "end_datetime"])

    visits_keys_pd = pd.read_parquet(
        str(OUT_ROOT / "visits"),
        columns=["person_id", "visit_id", "hospital"],
    ).drop_duplicates()

    def _filter_by_visits(pdf):
        return pdf.merge(visits_keys_pd, on=["person_id", "visit_id", "hospital"], how="inner")

    meas = meas.map_partitions(_filter_by_visits)

    out_path = OUT_ROOT / "measurement"
    # Hive-partition by hospital + sort by concept_id + small row groups, so
    # get_data()-style queries can prune both whole hospital folders and
    # row groups within a file, instead of reading every partition in full.
    _write_partitions(
        meas, out_path, label="Phase 2",
        partition_on=["hospital"], sort_cols=["concept_id"], row_group_size=50_000,
    )
    shutil.rmtree(_TMP, ignore_errors=True)
    # Restart distributed client now that measurement is done
    from dask.distributed import Client as _DClient
    client = _DClient(processes=True)
    print(f"measurement written to {out_path}")

## Run All

In [ ]:
if __name__ == "__main__":
    print("=== Building unified datasets ===")

    print("\n[1/5] Building visits...")
    build_visits()
    validate_visits()        # stops here if new data is missing
    client.restart(); gc.collect()

    print("\n[2/5] Building clinical_events...")
    build_clinical_events()
    client.restart(); gc.collect()

    print("\n[3/5] Building drug_exposure...")
    build_drug_exposure()
    client.restart(); gc.collect()

    print("\n[4/5] Building measurement...")
    build_measurement()
    client.restart(); gc.collect()

    print("\n[5/5] Building visit_types...")
    build_visit_types()
    validate_visit_types()   # stops here if care_site mapping is broken

    print("\n=== Done. All unified tables written to processed-data/ ===")


In [ ]:
clinical   = dd.read_parquet(OUT_ROOT / "clinical_events", engine="pyarrow")
summarize_table(
    "clinical_events",
    clinical,
    key_cols=["person_id", "visit_id", "origin", "concept_id", "concept_name", "start_datetime", "end_datetime", "value", "str_value"]
)


visits     = dd.read_parquet(OUT_ROOT / "visits", engine="pyarrow")
summarize_table(
    "visits",
    visits,
    key_cols=["person_id", "visit_id", "start_datetime", "end_datetime", "visit_type", "death_flag"]
)

drug       = dd.read_parquet(OUT_ROOT / "drug_exposure", engine="pyarrow")
summarize_table(
    "drug_exposure",
    drug,
    key_cols=["person_id", "visit_id", "concept_id", "concept_name", "drug_type_concept_id",
              "drug_type_category", "atc"]
)

meas       = dd.read_parquet(OUT_ROOT / "measurement", engine="pyarrow")
summarize_table(
    "measurement",
    meas,
    key_cols=["person_id", "concept_id", "concept_name", "unit", "start_datetime", "value"]
)


===== SUMMARY: clinical_events =====
Rows: 7294071

Dtypes:
person_id                 object
visit_id                  object
start_datetime    datetime64[ns]
end_datetime      datetime64[ns]
concept_id                object
concept_name              object
source_value              object
hospital                  object
origin                    object
value                    float64
str_value                 string
dtype: object

Column: person_id
  nulls: 0
  sample values: []

Column: visit_id
  nulls: 0
  sample values: []

Column: origin
  nulls: 0
  sample values: []

Column: concept_id
  nulls: 0
  sample values: []

Column: concept_name
  nulls: 0
  sample values: []

Column: start_datetime
  nulls: 0
  sample values: [Timestamp()]

Column: end_datetime
  nulls: 0
  sample values: [Timestamp()]

Column: value
  nulls: 7283829
  sample values: []

Column: str_value
  nulls: 
  sample values: []

===== SUMMARY: visits =====
Rows: 

Dtypes:
person_id                      objec

## Building Dimentions Lookup Tables

In [ ]:
DIM_ROOT.mkdir(exist_ok=True)

In [ ]:
def build_dim_clinical_concepts():
    clinical = dd.read_parquet(
        OUT_ROOT / "clinical_events",
        columns=["concept_id", "concept_name"]
    )

    grouped = (
        clinical
        .dropna(subset=["concept_id", "concept_name"])
        .groupby(["concept_id", "concept_name"])
        .size()
    )

    # Convert to pandas, rename the resulting column
    pdf = (
        grouped.compute()
        .reset_index()
        .rename(columns={0: "event_count"})
        .sort_values(["event_count"], ascending=[False])
    )

    pdf.to_parquet(DIM_ROOT / "clinical_concepts.parquet", index=False)
    return pdf

def build_dim_measurement_concepts():
    meas = dd.read_parquet(
        OUT_ROOT / "measurement",
        columns=["concept_id", "concept_name"]
    )

    grouped = (
        meas
        .dropna(subset=["concept_id", "concept_name"])
        .groupby(["concept_id", "concept_name"])
        .size()
    )

    # Convert to pandas, rename the resulting column
    pdf = (
        grouped.compute()
        .reset_index()
        .rename(columns={0: "event_count"})
        .sort_values(["event_count"], ascending=[False])
    )
    
    pdf.to_parquet(DIM_ROOT / "measurement_concepts.parquet", index=False)
    return pdf


def build_dim_measurement_concept_units():
    """
    Build a dimension table:
    (concept_id, concept_name, unit)
    -> event_count, value_min, value_mean, value_max
    """

    # 1. Read the measurement table
    meas = dd.read_parquet(
        OUT_ROOT / "measurement",
        columns=["concept_id", "concept_name", "unit", "value"],
    )

    # Keep rows with concept_id + concept_name
    meas = meas.dropna(subset=["concept_id", "concept_name"])

    # 2. Aggregate with Dask
    grouped = (
        meas.groupby(["concept_id", "concept_name", "unit"])
        .agg({
            "value": ["min", "mean", "max", "count"]
        })
    )

    # 3. Convert to pandas + clean column names
    pdf = grouped.compute()
    pdf.columns = ["value_min", "value_mean", "value_max", "event_count"]

    # 4. Sort for convenience
    pdf = pdf.sort_values("event_count", ascending=False)
    pdf = pdf.reset_index()

    # 5. Save
    pdf.to_parquet(DIM_ROOT / "measurement_concept_units.parquet", index=False)

    return pdf


def build_dim_drug_concepts():
    drug = dd.read_parquet(
        OUT_ROOT / "drug_exposure",
        columns=["drug_type_category", "concept_id", "concept_name"]
    )

    grouped = (
        drug
        .dropna(subset=["concept_id", "concept_name"])
        .groupby(["drug_type_category", "concept_id", "concept_name"])
        .size()
    )
    
    # Convert to pandas, rename the resulting column
    pdf = (
        grouped.compute()
        .reset_index()
        .rename(columns={0: "event_count"})
        .sort_values(
            ["drug_type_category", "event_count"], 
            ascending=[True, False]
        )
    )

    pdf.to_parquet(DIM_ROOT / "drug_concepts.parquet", index=False)
    return pdf


def build_dim_drug_routes():
    # Read only the columns we need from the unified drug_exposure table
    drug = dd.read_parquet(
        OUT_ROOT / "drug_exposure",
        columns=["drug_type_category", "concept_id", "concept_name", "route", "quantity"],
    )

    # Group by type / concept / route and aggregate quantity
    grouped = (
        drug
        .dropna(subset=["concept_id", "concept_name", "route"])
        .groupby(["drug_type_category", "concept_id", "concept_name", "route"])["quantity"]
        .agg(["count", "min", "mean", "max"])
    )

    # Bring to pandas, rename columns, sort
    pdf = (
        grouped.compute()
        .reset_index()
        .rename(
            columns={
                "count": "event_count",
                "min": "quantity_min",
                "mean": "quantity_avg",
                "max": "quantity_max",
            }
        )
        .sort_values(
            ["drug_type_category", "event_count"],
            ascending=[True, False],
        )
    )

    # Save the dimension table
    pdf.to_parquet(DIM_ROOT / "drug_routes.parquet", index=False)
    return pdf

In [ ]:
build_dim_clinical_concepts()
build_dim_measurement_concepts()
build_dim_measurement_concept_units()
build_dim_drug_concepts()
build_dim_drug_routes()

drug_type_category concept_id  \

                                            concept_name  \

                              route  event_count

# Query Tools

In [ ]:
def apply_pattern_filter(df: pd.DataFrame, col: str, pattern):
    """
    pattern can be:
      - string (simple contains)
      - string with % separators ? AND condition
      - list of strings ? OR across items
    """
    series = df[col].astype(str).str.lower()

    def match_and(pattern_str: str):
        """Return mask where ALL substrings appear."""
        parts = [p.strip().lower() for p in pattern_str.split("%") if p.strip()]
        mask = pd.Series(True, index=df.index)
        for part in parts:
            mask &= series.str.contains(part, na=False)
        return mask

    # Simple pattern: no OR
    if isinstance(pattern, str):
        # AND logic if % appears
        if "%" in pattern:
            return df[match_and(pattern)]
        else:
            return df[series.str.contains(pattern.lower(), na=False)]

    # OR logic if pattern is list/tuple
    if isinstance(pattern, (list, tuple)):
        combined_mask = pd.Series(False, index=df.index)
        for p in pattern:
            if "%" in p:
                combined_mask |= match_and(p)
            else:
                combined_mask |= series.str.contains(p.lower(), na=False)
        return df[combined_mask]

    raise ValueError("Invalid pattern type. Must be str or list of str.")

def format_lookup_rows(
    df: pd.DataFrame,
    id_col: str,
    label_cols,
    count_col: str = "event_count",
    aliases: dict = None,   # NEW
) -> str:
    """
    Turn a small pandas df into a multi-line string:
    ID: ... | COL1: ... | COL2: ... | COUNT: ...
    Optionally rename label headers via `aliases` dict.
    """
    if isinstance(label_cols, str):
        label_cols = [label_cols]

    aliases = aliases or {}

    lines = []
    for _, row in df.iterrows():
        labels = " | ".join(
            f"{aliases.get(col, col.upper())}: {row[col]}"
            for col in label_cols
        )
        line = f"ID: {row[id_col]} | {labels} | COUNT: {row[count_col]}"
        lines.append(line)

    return "\n".join(lines)

def load_dim(name: str) -> pd.DataFrame:
    return pd.read_parquet(DIM_ROOT / f"{name}.parquet")


DIM_CONFIG: Dict[str, Dict] = {
    "clinical": {
        "file": "clinical_concepts",
        "id_col": "concept_id",
        "default_label_cols": ["concept_name"],
        "main_label_col": "concept_name",
        "count_col": "event_count",
    },
    "measurement": {
        "file": "measurement_concepts",
        "id_col": "concept_id",
        "default_label_cols": ["concept_name"],
        "main_label_col": "concept_name",
        "count_col": "event_count",
    },
    "measurement_units": {
        "file": "measurement_concept_units",
        "id_col": "concept_id",
        "default_label_cols": ["concept_name", "unit"],
        "main_label_col": "concept_name",  # you could switch to "unit" if you prefer
        "count_col": "event_count",
    },
    "drug": {
        "file": "drug_concepts",
        "id_col": "concept_id",
        "default_label_cols": ["drug_type_category", "concept_name"],
        "main_label_col": "concept_name",
        "count_col": "event_count",
    },
    "drug_routes": {
        "file": "drug_routes",
        "id_col": "concept_id",
        "default_label_cols": ["drug_type_category", "concept_name", "route", 
                               "quantity_min", "quantity_avg", "quantity_max"],
        "main_label_col": "concept_name",
        "count_col": "event_count",
    },
}


def query_dim(
    dim: str,
    *,
    name_pattern: Optional[Union[str, List[str]]] = None,
    filters: Optional[Dict[str, Union[str, int, List[Union[str, int]]]]] = None,
    extra_label_cols: Optional[List[str]] = None,
    top_n: Optional[int] = None,
    return_format: str = "df",   # "df" or "json"
):
    """
    Generic lookup over dimension tables.

    dim:
        One of {"clinical", "measurement", "measurement_units",
                "drug", "drug_routes"}.

    name_pattern:
        Pattern(s) applied to the main label column (e.g. concept_name or route).
        - str        -> LIKE '%pattern%'
        - list/tuple -> OR of multiple patterns

    filters:
        Dict of equality filters on columns, e.g.
        {"drug_type_category": ["given_in_hospital", "discharge_perscription"]}

    extra_label_cols:
        Extra label columns to include in output (in addition to defaults).

    top_n:
        If given, return only first N rows after filtering & sorting.

    return_format:
        "df"   -> returns pandas DataFrame (ID + labels + count).
        "json" -> returns multi-line string using format_lookup_rows()
                  (each line includes ID, labels, COUNT).
    """
    if dim not in DIM_CONFIG:
        raise ValueError(f"Unknown dim '{dim}'. Valid options: {list(DIM_CONFIG.keys())}")

    cfg = DIM_CONFIG[dim]
    df = load_dim(cfg["file"])

    id_col = cfg["id_col"]
    count_col = cfg["count_col"]
    default_label_cols = list(cfg["default_label_cols"])
    main_label_col = cfg["main_label_col"]

    # 1) apply filters
    if filters:
        for col, val in filters.items():
            if col not in df.columns:
                raise ValueError(f"Filter column '{col}' not in dim '{dim}'")
            if isinstance(val, (list, tuple, set)):
                df = df[df[col].isin(list(val))]
            else:
                df = df[df[col] == val]

    # 2) apply name pattern on main label column
    if name_pattern is not None:
        if main_label_col not in df.columns:
            raise ValueError(f"Main label col '{main_label_col}' not in dim '{dim}'")
        df = apply_pattern_filter(df, main_label_col, name_pattern)

    # 3) sort by count descending when available
    if count_col in df.columns:
        df = df.sort_values(count_col, ascending=False)

    # 4) top_n
    if top_n is not None:
        df = df.head(top_n)

    # 5) determine label columns
    label_cols: List[str] = default_label_cols
    if extra_label_cols:
        for c in extra_label_cols:
            if c in df.columns and c not in label_cols:
                label_cols.append(c)

    # 6) return format
    if return_format == "df":
        cols = [c for c in [id_col] + label_cols + [count_col] if c in df.columns]
        return df[cols].reset_index(drop=True)

    if return_format == "json":
        # JSON mode is 1-column oriented: use main_label_col
        value_col = main_label_col
        missing = [c for c in (id_col, value_col, count_col) if c not in df.columns]
        if missing:
            raise ValueError(f"Missing expected columns {missing} in dim '{dim}'")

        df_small = df[[id_col, value_col, count_col]].reset_index(drop=True)

        col_name = value_col  # e.g. "concept_name" or "route"

        lines = []
        for _, row in df_small.iterrows():
            line = f"'ID: {row[id_col]} | Value: {row[value_col]} | COUNT: {row[count_col]}'"
            lines.append(line)

        inner = ",\n".join(lines)
        # Dict-like wrapper with the column name as key
        result = f"'{col_name}': {{\n{inner}\n}}"
        return result

    raise ValueError("return_format must be 'df' or 'json'")

In [ ]:
txt = query_dim(
    "drug",
    name_pattern=[
    # SGLT2 inhibitors
    "empagliflozin", "dapagliflozin", "canagliflozin", "ertugliflozin",
    "jardiance", "jardiance duo", "forxiga", "xigduo", "glyxambi"

    # Sulfonylureas
    "glibenclamide", "glyburide", "gliclazide", "glimepiride", "glipizide",
    "gliben", "glucorite", "amaryl",

    # Glinides
    "repaglinide", "nateglinide",
    "novonorm"
],
    filters={"drug_type_category":["given_in_hospital", "chronic_medications"]},
    # filters={},
    return_format="json",   # multi-line string via format_lookup_rows
)
print(txt)

'concept_name': {
'ID:  | Value:  | COUNT: ',
}


In [ ]:
txt = query_dim(
    "drug",
    name_pattern=[
    # SGLT2 inhibitors
    "metformin", ""],
    filters={"drug_type_category":["given_in_hospital", "chronic_medications"]},
    # filters={},
    return_format="json",   # multi-line string via format_lookup_rows
)
print(txt)

'concept_name': {
'ID:  | Value:  | COUNT: ',
'ID:  | Value:  | COUNT: ',
}


In [ ]:
res = query_dim(
    "measurement_units",
    name_pattern=[
    "glomerular"],
    filters={},
    top_n=80,
    return_format="df",
    extra_label_cols=["min", "avg", "max"],
)
# res[res["concept_id", "concept_name", "route"]]
res

concept_id                                       concept_name  \
0     
1     
2     
3    
4     
5     
6     

             unit  event_count

# DB Corrections / Normalizations

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

def load_norm_rules_from_dir(rules_dir: Path) -> List[Dict[str, Any]]:
    """
    Read all *.txt JSON rule files from a directory and return a flat list of rules.
    Each file contains a JSON list of rule dicts.
    """
    all_rules = []

    for path in Path(rules_dir).glob("*.txt"):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()

        text = re.sub(r"//.*?$", "", text, flags=re.MULTILINE)
        text = re.sub(r"/\*.*?\*/", "", text, flags=re.DOTALL)
        text = re.sub(r",\s*(\]|\})", r"\1", text)

        try:
            rules = json.loads(text)
        except Exception as e:
            print(f"\nFailed to parse rule file: {path}\n{text}\nError: {e}\n")
            raise

        for r in rules:
            if "old_unit" in r:
                r["old_unit"] = str(r["old_unit"]).lower().strip()
            if "new_unit" in r:
                r["new_unit"] = str(r["new_unit"]).lower().strip()

        all_rules.extend(rules)

    return all_rules


def normalize_measurements(
    meas_path,
    rules: List[Dict[str, Any]],
    out_path,
    npartitions: int = 200,
    track_counts: bool = True,
) -> Tuple[dd.DataFrame, Dict[int, int]]:
    """
    Apply unit/value normalization rules to the measurement table and write a new Parquet folder.

    Uses the synchronous scheduler so only one partition is processed at a time â€”
    avoids worker OOM on large measurement tables. Also means count tracking works
    correctly via a shared accumulator (same process, sequential execution).
    """
    ddf = dd.read_parquet(meas_path)

    # Pre-index rules by unit for fast per-partition lookup
    rules_by_unit: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for idx, r in enumerate(rules):
        r = dict(r)
        r["_idx"] = idx
        ou = r.get("old_unit_lower") or r.get("old_unit")
        if ou is None:
            key = "__any__"
        else:
            key = str(ou).lower()
            r["old_unit_lower"] = key
        rules_by_unit[key].append(r)
    rules_by_unit = dict(rules_by_unit)

    # Mutable accumulator â€” works because we use scheduler='synchronous' below,
    # so all partitions run sequentially in the same process
    rule_counts: Dict[int, int] = defaultdict(int) if track_counts else {}

    def _apply_rules(pdf):
        pdf = pdf.copy()
        unit_lower = pdf["unit"].astype(str).str.lower()
        n_rows = len(pdf)
        local_counts: Dict[int, int] = defaultdict(int)
        present_units = pd.unique(unit_lower)

        for u in present_units:
            u_str = str(u)
            unit_rules = rules_by_unit.get(u_str, [])
            if not unit_rules:
                continue
            base_mask_unit = unit_lower.eq(u_str).to_numpy()
            if not base_mask_unit.any():
                continue

            for rule in unit_rules:
                mask = base_mask_unit.copy()
                if "concept_id" in rule:
                    cid = rule["concept_id"]
                    if isinstance(cid, (list, tuple, set)):
                        mask &= pdf["concept_id"].isin(cid).to_numpy()
                    else:
                        mask &= (pdf["concept_id"] == cid).to_numpy()
                if not mask.any():
                    continue
                if "min_value" in rule:
                    mask &= pdf["value"].to_numpy() >= rule["min_value"]
                if "max_value" in rule:
                    mask &= pdf["value"].to_numpy() <= rule["max_value"]
                if not mask.any():
                    continue
                n_match = int(mask.sum())
                if n_match == 0:
                    continue

                if "expr" in rule:
                    x = pdf.loc[mask, "value"].astype("float64")
                    pdf.loc[mask, "value"] = eval(rule["expr"], {"np": np}, {"x": x})
                else:
                    if "value_factor" in rule:
                        pdf.loc[mask, "value"] = pdf.loc[mask, "value"].astype("float64") * rule["value_factor"]
                    if "value_offset" in rule:
                        pdf.loc[mask, "value"] = pdf.loc[mask, "value"].astype("float64") + rule["value_offset"]
                if "new_unit" in rule:
                    pdf.loc[mask, "unit"] = rule["new_unit"]
                local_counts[rule["_idx"]] += n_match

        if "__any__" in rules_by_unit:
            base_mask_any = np.ones(n_rows, dtype=bool)
            for rule in rules_by_unit["__any__"]:
                mask = base_mask_any.copy()
                if "concept_id" in rule:
                    cid = rule["concept_id"]
                    if isinstance(cid, (list, tuple, set)):
                        mask &= pdf["concept_id"].isin(cid).to_numpy()
                    else:
                        mask &= (pdf["concept_id"] == cid).to_numpy()
                if not mask.any():
                    continue
                if "min_value" in rule:
                    mask &= pdf["value"].to_numpy() >= rule["min_value"]
                if "max_value" in rule:
                    mask &= pdf["value"].to_numpy() <= rule["max_value"]
                if not mask.any():
                    continue
                n_match = int(mask.sum())
                if n_match == 0:
                    continue
                if "expr" in rule:
                    x = pdf.loc[mask, "value"].astype("float64")
                    pdf.loc[mask, "value"] = eval(rule["expr"], {"np": np}, {"x": x})
                else:
                    if "value_factor" in rule:
                        pdf.loc[mask, "value"] = pdf.loc[mask, "value"].astype("float64") * rule["value_factor"]
                    if "value_offset" in rule:
                        pdf.loc[mask, "value"] = pdf.loc[mask, "value"].astype("float64") + rule["value_offset"]
                if "new_unit" in rule:
                    pdf.loc[mask, "unit"] = rule["new_unit"]
                local_counts[rule["_idx"]] += n_match

        # Accumulate into shared dict â€” safe because synchronous scheduler runs this
        # function sequentially in the driver process, not in separate worker processes
        if track_counts:
            for k, v in local_counts.items():
                rule_counts[k] += v

        return pdf

    ddf_norm = ddf.map_partitions(_apply_rules)

    out_path = Path(out_path)

    # _write_partitions already processes one partition at a time on the driver
    # (avoids worker OOM) and keeps the rule_counts accumulator correct (same
    # process, sequential). Hive-partition by hospital + sort by concept_id +
    # small row groups so downstream queries can prune both.
    _write_partitions(
        ddf_norm, out_path, label="measurement_norm",
        partition_on=["hospital"], sort_cols=["concept_id"], row_group_size=50_000,
    )
    print(f"Normalized measurements written to {out_path}")

    return ddf_norm, dict(rule_counts)

In [ ]:
meas_rules = load_norm_rules_from_dir(RULES_DIR)

meas_norm_ddf, rule_counts = normalize_measurements(
    meas_path=OUT_ROOT / "measurement",
    rules=meas_rules,
    out_path=OUT_ROOT / "measurement_norm",
    npartitions=200,
    track_counts=True,
)

print("Rule usage:")
for idx, cnt in sorted(rule_counts.items()):
    print(f"  rule #{idx}: {cnt} rows")

/home/odedshah.post.ac.il/.local/lib/python3.7/site-packages/ipykernel_launcher.py:219: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
/home/odedshah.post.ac.il/.local/lib/python3.7/site-packages/ipykernel_launcher.py:219: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
/home/odedshah.post.ac.il/.local/lib/python3.7/site-packages/ipykernel_launcher.py:219: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access


Normalized measurements written to /home/odedshah.post.ac.il/Desktop/local_share/lynx-workspace/unified_data/measurement_norm
Rule usage:
  rule #0:  rows
  rule #1:  rows
  rule #3:  rows
  rule #4:  rows
  rule #5:  rows
  rule #6:  rows
  rule #7:  rows
  rule #8:  rows
  rule #9:  rows
  rule #10:  rows
  rule #11:  rows
  rule #12:  rows
  rule #13:  rows
  rule #14:  rows
  rule #15:  rows
  rule #16:  rows
  rule #17:  rows
  rule #18:  rows
  rule #19:  rows
  rule #20:  rows
  rule #21:  rows
  rule #22:  rows
  rule #23:  rows
  rule #24:  rows
  rule #25:  rows
  rule #26:  rows
  rule #27:  rows
  rule #28:  rows
  rule #29:  rows


In [ ]:
def build_dim_measurement_concepts_norm():
    meas = dd.read_parquet(
        OUT_ROOT / "measurement_norm",
        columns=["concept_id", "concept_name"]
    )

    grouped = (
        meas
        .dropna(subset=["concept_id", "concept_name"])
        .groupby(["concept_id", "concept_name"])
        .size()
    )

    # Convert to pandas, rename the resulting column
    pdf = (
        grouped.compute()
        .reset_index()
        .rename(columns={0: "event_count"})
        .sort_values(["event_count"], ascending=[False])
    )
    
    pdf.to_parquet(DIM_ROOT / "norm_measurement_concepts.parquet", index=False)
    return pdf


def build_dim_measurement_concept_units_norm():
    """
    Build a dimension table:
    (concept_id, concept_name, unit)
    -> event_count, value_min, value_mean, value_max
    """

    # 1. Read the measurement table
    meas = dd.read_parquet(
        OUT_ROOT / "measurement_norm",
        columns=["concept_id", "concept_name", "unit", "value"],
    )

    # Keep rows with concept_id + concept_name
    meas = meas.dropna(subset=["concept_id", "concept_name"])

    # 2. Aggregate with Dask
    grouped = (
        meas.groupby(["concept_id", "concept_name", "unit"])
        .agg({
            "value": ["min", "mean", "max", "count"]
        })
    )

    # 3. Convert to pandas + clean column names
    pdf = grouped.compute()
    pdf.columns = ["value_min", "value_mean", "value_max", "event_count"]

    # 4. Sort for convenience
    pdf = pdf.sort_values("event_count", ascending=False)
    pdf = pdf.reset_index()

    # 5. Save
    pdf.to_parquet(DIM_ROOT / "norm_measurement_concept_units.parquet", index=False)

    return pdf

In [ ]:
build_dim_measurement_concepts_norm()
build_dim_measurement_concept_units_norm()

concept_id                                       concept_name       unit  \
0                    mg/dl   
1      mg/dl   
2              mg/dl   
3                           g/dl   
4                      mg/dl   
...         ...                                                ...        ...   
                                   %   
                                %   
           mIU/ml   
            AU/ml   
        

      value_min   value_mean  value_max  event_count  
0                     
...         ...          ...        ...          ...  
                1  
                 1  
              1  
              1  
        

[ rows x 7 columns]

In [ ]:
DIM_CONFIG: Dict[str, Dict] = {
    "clinical": {
        "file": "clinical_concepts",
        "id_col": "concept_id",
        "default_label_cols": ["concept_name"],
        "main_label_col": "concept_name",
        "count_col": "event_count",
    },
    "measurement": {
        "file": "norm_measurement_concepts",
        "id_col": "concept_id",
        "default_label_cols": ["concept_name"],
        "main_label_col": "concept_name",
        "count_col": "event_count",
    },
    "measurement_units": {
        "file": "norm_measurement_concept_units",
        "id_col": "concept_id",
        "default_label_cols": ["concept_name", "unit"],
        "main_label_col": "concept_name",  # you could switch to "unit" if you prefer
        "count_col": "event_count",
    },
    "drug": {
        "file": "drug_concepts",
        "id_col": "concept_id",
        "default_label_cols": ["drug_type_category", "concept_name"],
        "main_label_col": "concept_name",
        "count_col": "event_count",
    },
    "drug_routes": {
        "file": "drug_routes",
        "id_col": "concept_id",
        "default_label_cols": ["drug_type_category", "concept_name", "route", 
                               "quantity_min", "quantity_avg", "quantity_max"],
        "main_label_col": "concept_name",
        "count_col": "event_count",
    },
}

In [ ]:
res = query_dim(
"measurement_units",
# name_pattern=[""],
# filters={"concept_id": [""]},

filters={"concept_id": ["", "", "", ""]},
top_n=10,
return_format="df",
)
res

concept_id                                       concept_name    unit  \
0       
1    
2     
3    
4     
5      
6    
7   

   event_count  
0          
1          
2          
3          
4             
5             
6              
7

In [ ]:
# Pre-flight: rewrite oversized measurement row groups in-place
# Rewrites source parquet files that have row groups exceeding RAM budget.
# Original file is kept as .bak before overwrite.
import pyarrow.parquet as pq
import gc, shutil

LARGE_RG_THRESHOLD_MB = 200
TARGET_ROW_GROUP_SIZE = 50_000

print("=== Scanning measurement row groups across all hospitals ===")
to_rewrite = []

for root in [RAW_ROOT_OLD, RAW_ROOT_NEW]:
    for hosp_dir in sorted(root.iterdir()):
        if not hosp_dir.is_dir():
            continue
        meas_dir = hosp_dir / "measurement"
        if not meas_dir.exists():
            continue
        for fpath in sorted(meas_dir.rglob("*.parquet")):
            pf = pq.ParquetFile(str(fpath))
            large_rgs = [
                pf.metadata.row_group(rg).total_byte_size / 1e6
                for rg in range(pf.num_row_groups)
                if pf.metadata.row_group(rg).total_byte_size > LARGE_RG_THRESHOLD_MB * 1e6
            ]
            if large_rgs:
                rel = fpath.relative_to(root.parent)
                print(f"  LARGE  {rel}  max={max(large_rgs):.0f} MB")
                to_rewrite.append(fpath)

print(f"{len(to_rewrite)} file(s) to rewrite.")

for fpath in to_rewrite:
    bak = fpath.with_suffix(".parquet.bak")
    if bak.exists():
        print(f"  SKIP (already rewritten): {fpath.relative_to(root.parent)}")
        continue
    rel = fpath.relative_to(RAW_ROOT_OLD.parent if fpath.is_relative_to(RAW_ROOT_OLD.parent) else RAW_ROOT_NEW.parent)
    print(f"  {rel} ...")
    table = pq.read_table(str(fpath))
    shutil.copy2(str(fpath), str(bak))
    pq.write_table(table, str(fpath), row_group_size=TARGET_ROW_GROUP_SIZE)
    del table
    gc.collect()
    print(f"  done")

print("=== All done. Safe to re-run build_measurement() ===")
